# 03. DEG解析 (発現変動遺伝子解析)カウント行列からDEG（Differentially Expressed Genes）を検出します。**カーネル**: RNA-seq (R) / rnaseq_r  **ツール**: DESeq2 / edgeR (両方)  **入力**: `results/count_matrix.csv`, `sample_metadata.csv`  **出力**: `results/deg_*.csv`, QCプロット

## 設定

In [ ]:
# === 設定 ===
LFC_THRESHOLD <- 1.0
PADJ_THRESHOLD <- 0.05
DEG_TOOL <- "deseq2"  # "deseq2" or "edger"

# 前フィルタリングパラメータ
MIN_COUNT_THRESHOLD <- 10   # 最小カウント数

dir.create("results", showWarnings = FALSE)
cat("DEG解析ツール:", DEG_TOOL, "\n")

## ライブラリ読み込み

In [ ]:
required_packages <- c("DESeq2", "edgeR", "ggplot2", "pheatmap", "dplyr", "readr", "tidyr", "RColorBrewer")
missing <- required_packages[!sapply(required_packages, requireNamespace, quietly = TRUE)]
if (length(missing) > 0) {
    stop(paste0(
        "以下のRパッケージがインストールされていません:\n  ",
        paste(missing, collapse = ", "),
        "\n\nconda環境が有効か確認してください:\n  conda activate rnaseq_env"
    ))
}

library(DESeq2)
library(edgeR)
library(ggplot2)
library(pheatmap)
library(dplyr)
library(readr)
library(tidyr)
library(RColorBrewer)
cat("全パッケージの読み込み完了\n")

## データ読み込み

In [ ]:
# カウント行列
count_data <- read.csv("results/count_matrix.csv", row.names = 1, check.names = FALSE)

# メタデータ
sample_info <- read.csv("sample_metadata.csv")
rownames(sample_info) <- sample_info$sample_id

# --- 条件名のバリデーション ---
# スペースや特殊文字が含まれるとファイル出力やRの処理でエラーになるため確認
conditions <- unique(sample_info$condition)
has_space <- grepl("\\s", conditions)
if (any(has_space)) {
  bad <- conditions[has_space]
  stop(paste0(
    "条件名にスペースが含まれています: ", paste(bad, collapse=", "),
    "\nスペースをアンダースコア(_)に置換してください。",
    "\n例: 'compound A 0.5 nM' -> 'CompA_0.5nM'"
  ))
}
cat("条件名チェック: OK\n")

# 順序を合わせる
count_data <- count_data[, sample_info$sample_id]

cat("カウント行列:", nrow(count_data), "genes x", ncol(count_data), "samples\n")
cat("条件:", paste(conditions, collapse = ", "), "\n")
cat("各条件のサンプル数:\n")
print(table(sample_info$condition))

# 比較数の表示
n_comp <- choose(length(conditions), 2)
cat("\n全ペアワイズ比較数:", n_comp, "組\n")

## 前フィルタリング

In [ ]:
min_group_size <- min(table(sample_info$condition))
keep <- rowSums(count_data >= MIN_COUNT_THRESHOLD) >= min_group_size
count_data <- count_data[keep, ]
cat("フィルタ後:", nrow(count_data), "genes\n")
cat("(閾値: カウント >=", MIN_COUNT_THRESHOLD, "のサンプルが", min_group_size, "以上)\n")

## DEG解析関数の定義

### DESeq2

In [ ]:
run_deseq2 <- function(counts, coldata, pair) {
  dds <- DESeqDataSetFromMatrix(
    countData = round(counts),
    colData = coldata,
    design = ~ condition
  )
  dds$condition <- relevel(dds$condition, ref = pair[2])
  dds <- DESeq(dds)
  
  res <- results(dds, contrast = c("condition", pair[1], pair[2]), alpha = 0.05)
  res_df <- as.data.frame(res)
  res_df$gene_id <- rownames(res_df)
  res_df$comparison <- paste0(pair[1], "_vs_", pair[2])
  
  return(list(dds = dds, results = res_df))
}

### edgeR

In [ ]:
run_edger <- function(counts, coldata, pair) {
  idx <- coldata$condition %in% pair
  sub_counts <- counts[, idx]
  sub_coldata <- coldata[idx, ]
  
  group <- factor(sub_coldata$condition, levels = c(pair[2], pair[1]))
  y <- DGEList(counts = sub_counts, group = group)
  y <- calcNormFactors(y)
  
  design <- model.matrix(~ group)
  y <- estimateDisp(y, design)
  fit <- glmQLFit(y, design)
  qlf <- glmQLFTest(fit, coef = 2)
  
  res <- topTags(qlf, n = Inf, sort.by = "none")$table
  res$gene_id <- rownames(res)
  res$comparison <- paste0(pair[1], "_vs_", pair[2])
  
  colnames(res)[colnames(res) == "logFC"] <- "log2FoldChange"
  colnames(res)[colnames(res) == "PValue"] <- "pvalue"
  colnames(res)[colnames(res) == "FDR"] <- "padj"
  
  return(list(fit = fit, results = res))
}

## 全ペアワイズ比較の実行

In [ ]:
conditions <- unique(sample_info$condition)
comparisons <- combn(conditions, 2, simplify = FALSE)

all_results <- list()

for (pair in comparisons) {
  comp_name <- paste0(pair[1], "_vs_", pair[2])
  cat("比較実行中:", comp_name, "\n")
  
  if (DEG_TOOL == "deseq2") {
    result <- run_deseq2(count_data, sample_info, pair)
  } else {
    result <- run_edger(count_data, sample_info, pair)
  }
  
  all_results[[comp_name]] <- result$results
  
  # 個別CSV出力
  write.csv(result$results, 
            file = paste0("results/deg_", comp_name, ".csv"),
            row.names = FALSE)
  
  # DEG数を表示
  n_up <- sum(result$results$padj < 0.05 & 
              result$results$log2FoldChange > 1.0, na.rm = TRUE)
  n_down <- sum(result$results$padj < 0.05 & 
                result$results$log2FoldChange < -1.0, na.rm = TRUE)
  cat("  Up:", n_up, " Down:", n_down, "\n")
}

# 全結果を統合
all_deg <- do.call(rbind, all_results)
write.csv(all_deg, "results/all_deg_results.csv", row.names = FALSE)
cat("\n全比較結果を results/all_deg_results.csv に出力しました\n")

## QC: PCAプロット

In [ ]:
# DESeq2使用時のPCA
dds_all <- DESeqDataSetFromMatrix(
  countData = round(count_data),
  colData = sample_info,
  design = ~ condition
)
vsd <- vst(dds_all, blind = TRUE)

pca_data <- plotPCA(vsd, intgroup = "condition", returnData = TRUE)
pv <- round(100 * attr(pca_data, "percentVar"))

p <- ggplot(pca_data, aes(PC1, PC2, color = condition)) +
  geom_point(size = 4) +
  xlab(paste0("PC1: ", pv[1], "% variance")) +
  ylab(paste0("PC2: ", pv[2], "% variance")) +
  theme_minimal(base_size = 14) +
  ggtitle("PCA - 全サンプル")

print(p)
ggsave("results/pca_plot.pdf", p, width = 8, height = 6)

## 完了- DEG結果が `results/deg_*_vs_*.csv` に出力されました- 統合結果: `results/all_deg_results.csv`- 次のノートブック `04_Visualization.ipynb` に進んでください- **Pythonカーネルに戻す**ことを忘れないでください